# Polynomial Regression으로 Calories_Burned 예측
* 성별 분리 (M/F)
* 원본 5개 변수 
* (Exercise_Duration, BPM, Temp_diff, Age, Weight(lb))

* Polynomial degree=3 + Ridge

* interaction 피처나 Height 같은 추가 변수는 오히려 노이즈

최종 RMSE : 0.17 

리더보드 : 0.15

## 1. 라이브러리 임포트, 랜덤 시드 , 상수 고정

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 2. 데이터 로드

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

## 3. Feature Engineering Function - 수정금지

In [3]:
def create_features_safe(df):
    df = df.copy()

    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6

    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'],
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)

    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']

    # 최종 TOP5에 쓰는 핵심 변수명 고정
    df["Intensity"] = df["Duration_x_BPM"]
    df["Effort"] = df["Weight(lb)"] * df["Intensity"]

    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']

    df['BPM_per_Duration'] = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)

    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())

    df['Weight_x_Duration'] = df['Weight(lb)'] * df['Exercise_Duration']

    df['Log_BPM'] = np.log1p(df['BPM'])
    df['Log_Duration'] = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur'] = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])

    # # 저칼로리 구간: 짧은 Duration 정밀도 향상
    # df['Duration_sqrt'] = np.sqrt(df['Exercise_Duration'])   # 1분↔2분↔3분 차이 증폭
    # df['Duration_cbrt'] = np.cbrt(df['Exercise_Duration'])   # 더 극단적 짧은 운동 강조

    # # 고칼로리 구간: 고체중 + 저연령 조합 포착
    # df['Weight_div_Age'] = df['Weight(lb)'] / (df['Age'] + 1)
    # df['Effort_div_Age'] = df['Effort'] / (df['Age'] + 1)   
    return df

### Preprocess (train-fit encoder) + Reduced feature 
최종 학습에 사용할 feature 집합 정리

* generated_cols는 파생변수 목록

* reduced는 base(원핫 포함) + TOP5

In [4]:
# 0) raw split (타겟 분리 원본피처만 남기기) - 모델 입력은 X와 y로 분리되어야 하고 ID는 예측에 필요 없기 때문
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw  = test.drop(['ID'], axis=1).copy()

# 1) feature engineering
before_cols = list(train_x_raw.columns)
train_x_feat = create_features_safe(train_x_raw)
test_x_feat  = create_features_safe(test_x_raw)

generated_cols = [c for c in train_x_feat.columns if c not in before_cols]  # 파생변수만

# 2) one-hot (train fit -> test transform)
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x_feat.columns]
if len(cat_cols) > 0:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x_feat[cat_cols]) # train에서만 fit 누수방지

    tr_cat = enc.transform(train_x_feat[cat_cols])
    te_cat = enc.transform(test_x_feat[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)

    train_x_feat = train_x_feat.drop(columns=cat_cols)
    test_x_feat  = test_x_feat.drop(columns=cat_cols)

    train_x_feat[cat_names] = tr_cat
    test_x_feat[cat_names]  = te_cat

train_x = train_x_feat.copy()
test_x  = test_x_feat.copy()

# 3) reduced = base + TOP5 (Pruning)
TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity'] # 중요도기반 5개만
base_feats = [c for c in train_x.columns if c not in generated_cols]  # 원핫 포함 base 유지 (원본 + 원핫)
# 추가 피처 명시
#NEW_FEATS  = ['Duration_sqrt', 'Duration_cbrt',   # 저칼로리 구간용
#               'Weight_div_Age', 'Effort_div_Age']  # 고칼로리 구간용

keep_cols = base_feats + TOP5_FIXED  #+ NEW_FEATS   
train_red = train_x[keep_cols].copy()
test_red  = test_x[keep_cols].copy()

print("Final train_x/test_x:", train_x.shape, test_x.shape)
print("Reduced train/test:", train_red.shape, test_red.shape)
print("TOP5 present:", [c for c in TOP5_FIXED if c in train_red.columns])

Final train_x/test_x: (7500, 28) (7500, 28)
Reduced train/test: (7500, 15) (7500, 15)
TOP5 present: ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']


## Train + Fine-tuned alpha + weight (최종)
KFold OOF + Test fold-averaging

	1.	OOF RMSE 계산 - 진짜 일반화 성능을 보기 위해(누수 방지)
	2.	Test 예측 생성 - fold마다 학습한 모델의 test 예측을 평균내서 더 안정화
	3.	두 베이스 모델을 섞어서 최종 예측 - identity 타겟 모델 + sqrt 타겟 모델을 convex weight로 결합


In [5]:
DEGREE = 3  # PolynomialFeatures로 3차까지 확장 (곡선 관계 표현)
ALPHA_ID = 0.05 # identity(원단위) Ridge 규제 강도 : 작을수록 규제가 약해져 표현력이 증가  pruning 후 약하게 하는 게 최적이었음
ALPHA_SQ = 0.009    # sqrt 타겟 Ridge 규제 강도 보조역할로 넣은거라 값 작음
W_ID = 0.9851676708561896   # 최종 결합 가중치(거의 identity 위주)

# 타겟 준비
y = train_y.values.astype(float)    #원단위 타겟
y_sqrt = np.sqrt(np.clip(y, 0, None))   #sqrt 변환 타겟 
            # clip 음수 방지(혹시몰라 안전장치용)        #sqrt는 큰 값(고칼로리) 쪽을 상대적으로 눌러서 학습하게 만들어, 일부 구간에서 안정화를 기대하는 변환

# OOF/Test 예측 배열 생성
# OOF는 fold별 validation index에만 채워지고, test는 fold마다 예측해서 평균냄
id_test = np.zeros(len(test_red))   #  identity 모델의 test 예측 누적(나중에 fold 평균)
sq_test = np.zeros(len(test_red))   # sqrt 모델의 test 예측 누적
id_oof = np.zeros(len(train_red))   #  identity 모델의 OOF 예측 저장 공간 (train 길이)
sq_oof = np.zeros(len(train_red))   # sqrt 모델의 OOF 예측 저장 공간

# sample_weight = np.ones(len(y))
# sample_weight[y <= 30]  = 3.0   # 저칼로리 103개 오류 → 집중
# sample_weight[y > 200]  = 3.0   # 고칼로리  27개 오류 → 집중

# KFold 루프 - 이 구조 덕분에 OOF가 누수 없이 생성됨
for tr_idx, va_idx in kf.split(train_red):
    #fold 데이터 분리
    X_tr_raw = train_red.iloc[tr_idx]   # train_red: 이미 (base + TOP5)로 pruning된 최종 입력, fold마다 train/val로 나눔
    X_va_raw = train_red.iloc[va_idx]

    # PolynomialFeatures (fold 내부 fit)
    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr = poly.fit_transform(X_tr_raw) # fit_transform은 오직 X_tr_raw에서만! 누수방지용
    X_va = poly.transform(X_va_raw)
    X_te = poly.transform(test_red)

    # StandardScaler (fold 내부 fit) 스케일러도 train fold에서만 fit
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_te)

    # Base 모델 1: identity Ridge
    m_id = Ridge(alpha=ALPHA_ID, random_state=RANDOM_STATE)
    m_id.fit(X_tr_s, y[tr_idx])#,sample_weight=sample_weight[tr_idx]) # 학습 타겟: 원단위 y

    id_oof[va_idx] = m_id.predict(X_va_s)   # OOF 예측: validation index 위치에 저장
    id_test += m_id.predict(X_te_s) / N_SPLITS  # test 예측: fold마다 평균 → / N_SPLITS로 누적

    # Base 모델 2: sqrt Ridge
    m_sq = Ridge(alpha=ALPHA_SQ, random_state=RANDOM_STATE)
    m_sq.fit(X_tr_s, y_sqrt[tr_idx])#,sample_weight=sample_weight[tr_idx])    # 	학습 타겟: sqrt(y)

    # 예측값은 sqrt 스케일이니까 다시 원단위로 복원해야 함
    pred_va_sq = m_sq.predict(X_va_s)
    pred_te_sq = m_sq.predict(X_te_s)
                        # clip은 음수 예측 방지(제곱하면 이상해질 수 있음) 복원은 **2
    sq_oof[va_idx] = np.clip(pred_va_sq, 0, None) ** 2
    sq_test += (np.clip(pred_te_sq, 0, None) ** 2) / N_SPLITS

# 칼로리는 음수가 있을 수 없으니 안전하게 0 이상으로 고정
id_oof = np.clip(id_oof, 0, None)
sq_oof = np.clip(sq_oof, 0, None)
id_test = np.clip(id_test, 0, None)
sq_test = np.clip(sq_test, 0, None)

# 최종 결합 (가중 합)\
# W_ID ≈ 0.985라서 거의 identity가 메인이지만, sqrt가 아주 미세하게 보정 (특정 구간에서 RMSE를 조금 깎아줌)
final_oof = np.clip(W_ID * id_oof + (1 - W_ID) * sq_oof, 0, None)
final_test = np.clip(W_ID * id_test + (1 - W_ID) * sq_test, 0, None)

# 정수화 (반올림) 후처리
final_oof_round = np.rint(final_oof)
final_test_round = np.rint(final_test)

# 혹시 음수/범위 방지
final_oof_round = np.clip(final_oof_round, 0, None)
final_test_round = np.clip(final_test_round, 0, None)

print("OOF RMSE float :", rmse(y, final_oof))
print("OOF RMSE round :", rmse(y, final_oof_round))

print("Test stats round mean/min/max:",
        float(final_test_round.mean()),
        float(final_test_round.min()),
        float(final_test_round.max()))

OOF RMSE float : 0.29936955597925347
OOF RMSE round : 0.22090722034374521
Test stats round mean/min/max: 89.70306666666667 1.0 313.0


In [6]:
final_test = final_test_round
submission["Calories_Burned"] = final_test
submission.to_csv("submit_final_02209.csv", index=False)

print("After  round:", rmse(y,final_test_round))
print(" 저장 : submit_final_02209.csv")

After  round: 89.32878446129968
 저장 : submit_final_02209.csv


## 최종 RMSE : 0.22090722034374521

## 제출 후 리더보드 점수 : 0.21571

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

y_true = train_y.values.astype(float)

# --- baseline: round-only ---
base_oof_int = np.rint(np.clip(final_oof, 0, None))
base_test_int = np.rint(np.clip(final_test, 0, None))

print("Baseline OOF RMSE (round-only):", rmse(y_true, base_oof_int))

# --- rule adjust function ---
def rule_adjust(pred_float, ref_float, eps):
    """
    pred_float: 최종 float 예측(final_oof/final_test)
    ref_float : 보정의 기준이 될 독립 추정치(추천: sq_oof/sq_test)
    eps       : 0.5 경계 주변 폭 (예: 0.02면 0.48~0.52만 보정)
    """
    pred_float = np.clip(pred_float, 0, None)
    ref_float = np.clip(ref_float, 0, None)

    lo = np.floor(pred_float)
    hi = lo + 1

    frac = pred_float - lo
    mask = np.abs(frac - 0.5) <= eps  # 경계 근처만

    out = np.rint(pred_float)  # 기본은 round-only

    # 경계 근처에서는 ref에 더 가까운 정수 선택
    choose_hi = np.abs(ref_float - hi) < np.abs(ref_float - lo)
    out[mask] = np.where(choose_hi[mask], hi[mask], lo[mask])

    return np.clip(out, 0, None)

# --- eps grid search on OOF ---
eps_grid = np.linspace(0.0, 0.06, 31)  # 0~0.06 (0.002 간격)
best = {"rmse": 1e18, "eps": None}

for eps in eps_grid:
    oof_adj = rule_adjust(final_oof, sq_oof, eps)  #  ref= sq_oof 추천
    score = rmse(y_true, oof_adj)
    if score < best["rmse"]:
        best = {"rmse": float(score), "eps": float(eps)}

print("\n BEST rule-adjust:", best)

# --- apply to test with best eps ---
final_test_rule = rule_adjust(final_test, sq_test, best["eps"])

# 제출 저장
sub = pd.read_csv("sample_submission.csv")
sub["Calories_Burned"] = final_test_rule.astype(int)
out_name = "submit_rule_plusminus1.csv"
sub.to_csv(out_name, index=False)
print("saved:", out_name)
print("test stats mean/min/max:", float(final_test_rule.mean()), float(final_test_rule.min()), float(final_test_rule.max()))

Baseline OOF RMSE (round-only): 0.22090722034374521

✅ BEST rule-adjust: {'rmse': 0.21847959477565254, 'eps': 0.004}
saved: submit_rule_plusminus1.csv
test stats mean/min/max: 89.70306666666667 1.0 313.0


타겟이 정수라는 점에 착안해, round-only를 적용한 뒤 라운딩 경계(0.5±0.004)에서만 보조 예측(sqrt base)으로 tie-break하는 규칙 기반 보정을 추가했고 OOF RMSE를 0.218까지 개선

In [21]:
cols = ['Age', 'Exercise_Duration', 'BPM', 'Weight(lb)', 'Body_Temperature(F)']
for col in cols:
    corr = train[col].corr(train['Calories_Burned'])
    print(f"{col}: {corr:.4f}")

Age: 0.1596
Exercise_Duration: 0.9548
BPM: 0.8999
Weight(lb): 0.0426
Body_Temperature(F): 0.8244


## 성별 분리 해서 시도

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import lstsq

# 공통 피처 준비 함수
def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    d['Height_Total'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    return d

core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total'] # 중요 피처
DEGREE = 5  # 일단 5로  () 6개 변수의 모든 5차 조합 생성용)

train_prep = prep_df(train)
test_prep  = prep_df(test)

# 성별 분리
male_mask2   = train_prep['Gender'] == 'M'
female_mask2 = ~male_mask2
male_test    = test_prep['Gender'] == 'M'
female_test  = ~male_test

results = {}
for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)

    X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
    X_te_b = np.column_stack([X_te_p, np.ones(len(X_te_p))])

    c, _, _, _ = lstsq(X_tr_b, y_tr, rcond=None) # lstsq (최소제곱법) 으로 계수 학습 — Ridge 없이 순수 선형회귀
    pred_tr = X_tr_b @ c
    pred_te = X_te_b @ c

    print(f"Gender={gender} RMSE: {rmse(y_tr, pred_tr):.4f}  "
            f"round: {rmse(y_tr, np.rint(np.clip(pred_tr,0,None))):.4f}") 
    results[gender] = {'c': c, 'poly': poly, 'pred_tr': pred_tr, 'pred_te': pred_te,
                        'mask': mask, 'test_mask': test_mask}

# 전체 OOF RMSE
pred_all = np.zeros(len(train))
pred_all[male_mask2.values]   = results['M']['pred_tr']
pred_all[female_mask2.values] = results['F']['pred_tr']
print(f"\n전체 RMSE float: {rmse(train['Calories_Burned'].values, pred_all):.4f}")
print(f"전체 RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

# 제출 파일
test_pred_all = np.zeros(len(test))
test_pred_all[male_test.values]   = results['M']['pred_te']
test_pred_all[female_test.values] = results['F']['pred_te']
test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int) # test 예측 후 반올림해서 저장용

# sub = pd.read_csv("sample_submission.csv")
# sub['Calories_Burned'] = test_pred_round
# sub.to_csv("submit_poly5_gender.csv", index=False)
# print("\n저장: submit_poly5_gender.csv")
# print(f"test 예측 mean/min/max: {test_pred_round.mean():.1f} / {test_pred_round.min()} / {test_pred_round.max()}")

Gender=M RMSE: 0.2751  round: 0.1717
Gender=F RMSE: 0.2726  round: 0.1716

전체 RMSE float: 0.2738
전체 RMSE round: 0.1717

저장: submit_poly5_gender.csv
test 예측 mean/min/max: 89.7 / 1 / 314


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import RidgeCV
import numpy as np

def prep_df(df):
    d = df.copy()
    d['Temp_diff'] = d['Body_Temperature(F)'] - 98.6
    d['Height_Total'] = d['Height(Feet)'] * 12 + d['Height(Remainder_Inches)']
    return d

core_cols = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)', 'Height_Total']

train_prep = prep_df(train)
test_prep  = prep_df(test)

male_mask2   = train_prep['Gender'] == 'M'
female_mask2 = ~male_mask2
male_test    = test_prep['Gender'] == 'M'
female_test  = ~male_test

pred_all      = np.zeros(len(train))
test_pred_all = np.zeros(len(test))

for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)          # train fit → test transform only

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)          # train fit → test transform only

    model = RidgeCV(alphas=np.logspace(-4, 2, 100))
    model.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  best alpha: {model.alpha_:.5f}  "
            f"train RMSE round: {rmse(y_tr, np.rint(np.clip(model.predict(X_tr_s),0,None))):.4f}")

    pred_all[mask.values]          = model.predict(X_tr_s)
    test_pred_all[test_mask.values] = model.predict(X_te_s)

print(f"\n전체 train RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

# test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
# sub = pd.read_csv("sample_submission.csv")
# sub['Calories_Burned'] = test_pred_round
# sub.to_csv("submit_ridge_poly5_final.csv", index=False)
# print("저장: submit_ridge_poly5_final.csv")

Gender=M  best alpha: 0.00081  train RMSE round: 0.1596
Gender=F  best alpha: 0.00327  train RMSE round: 0.1537

전체 train RMSE round: 0.1566
저장: submit_ridge_poly5_final.csv


# RMSE 0.1566 리더보드 0.2019900988

# 어ㅏㅏㅗ리ㅏㅑㅇ노리ㅏㅑㄴ몰;ㅑ농;랴ㅗㅁㅇㄴ

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# =========================
# 0) 설정
# =========================
RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

DEGREE = 3
ALPHA_SQ = 0.009

TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']

alpha_grid = np.linspace(0.02, 0.10, 41)      # alpha_id 후보
w_grid     = np.linspace(0.95, 1.00, 101)     # weight 후보
eps_grid   = np.linspace(0.0, 0.010, 51)      # rule-adjust eps 후보

# =========================
# 1) 원본에서 다시 시작 (상태 꼬임 방지)
# =========================
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw  = test.drop(['ID'], axis=1).copy()

before_cols = list(train_x_raw.columns)

train_x_feat = create_features_safe(train_x_raw)
test_x_feat  = create_features_safe(test_x_raw)

generated_cols = [c for c in train_x_feat.columns if c not in before_cols]

# =========================
# 2) 원핫 인코딩 (train fit -> test transform)
# =========================
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x_feat.columns]
if len(cat_cols) > 0:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x_feat[cat_cols])

    tr_cat = enc.transform(train_x_feat[cat_cols])
    te_cat = enc.transform(test_x_feat[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)

    train_x_feat = train_x_feat.drop(columns=cat_cols)
    test_x_feat  = test_x_feat.drop(columns=cat_cols)

    train_x_feat[cat_names] = tr_cat
    test_x_feat[cat_names]  = te_cat

train_x_final = train_x_feat.copy()
test_x_final  = test_x_feat.copy()

# =========================
# 3) (중요) keep_cols 안전 구성: 존재하는 컬럼만 사용
# =========================
# base_feats = 파생변수 제외한 "원본+원핫"만
base_feats = [c for c in train_x_final.columns if c not in generated_cols]

# TOP5 중 실제 존재하는 것만
top5_exists = [c for c in TOP5_FIXED if c in train_x_final.columns]
missing_top5 = [c for c in TOP5_FIXED if c not in train_x_final.columns]

print("TOP5 exists:", top5_exists)
print("TOP5 missing:", missing_top5)

keep_cols_raw = base_feats + top5_exists
keep_cols = [c for c in keep_cols_raw if c in train_x_final.columns]

# train/test 컬럼 정렬 (혹시 test에 없는 컬럼 있으면 0 채움)
train_red = train_x_final.reindex(columns=keep_cols, fill_value=0).copy()
test_red  = test_x_final.reindex(columns=keep_cols,  fill_value=0).copy()
train_red, test_red = train_red.align(test_red, join="left", axis=1, fill_value=0)

y = train_y.values.astype(float)
y_sqrt = np.sqrt(np.clip(y, 0, None))

print("Final train/test shape:", train_x_final.shape, test_x_final.shape)
print("Reduced train/test shape:", train_red.shape, test_red.shape)

# =========================
# 4) fold cache (poly+scaler)
# =========================
fold_cache = []
for tr_idx, va_idx in kf.split(train_red):
    X_tr_raw = train_red.iloc[tr_idx]
    X_va_raw = train_red.iloc[va_idx]

    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr = poly.fit_transform(X_tr_raw)
    X_va = poly.transform(X_va_raw)
    X_te = poly.transform(test_red)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_te)

    fold_cache.append((tr_idx, va_idx, X_tr_s, X_va_s, X_te_s))

# =========================
# 5) rule-adjust (경계 근처 tie-break)
# =========================
def rule_adjust(pred_float, ref_float, eps):
    pred_float = np.clip(pred_float, 0, None)
    ref_float = np.clip(ref_float, 0, None)

    lo = np.floor(pred_float)
    hi = lo + 1
    frac = pred_float - lo

    out = np.rint(pred_float)
    mask = np.abs(frac - 0.5) <= eps

    choose_hi = np.abs(ref_float - hi) < np.abs(ref_float - lo)
    out[mask] = np.where(choose_hi[mask], hi[mask], lo[mask])
    return np.clip(out, 0, None)

# =========================
# 6) 튜닝: alpha_id -> w -> eps (OOF round RMSE 최소화)
# =========================
best = {"rmse": 1e18, "alpha_id": None, "w": None, "eps": None}
best_artifacts = None

for alpha_id in tqdm(alpha_grid, desc="alpha_id search"):
    id_oof = np.zeros(len(train_red))
    id_test = np.zeros(len(test_red))
    sq_oof = np.zeros(len(train_red))
    sq_test = np.zeros(len(test_red))

    for tr_idx, va_idx, X_tr_s, X_va_s, X_te_s in fold_cache:
        # identity
        m_id = Ridge(alpha=float(alpha_id), random_state=RANDOM_STATE)
        m_id.fit(X_tr_s, y[tr_idx])
        id_oof[va_idx] = m_id.predict(X_va_s)
        id_test += m_id.predict(X_te_s) / N_SPLITS

        # sqrt
        m_sq = Ridge(alpha=float(ALPHA_SQ), random_state=RANDOM_STATE)
        m_sq.fit(X_tr_s, y_sqrt[tr_idx])
        pred_va_sq = m_sq.predict(X_va_s)
        pred_te_sq = m_sq.predict(X_te_s)
        sq_oof[va_idx] = np.clip(pred_va_sq, 0, None) ** 2
        sq_test += (np.clip(pred_te_sq, 0, None) ** 2) / N_SPLITS

    id_oof = np.clip(id_oof, 0, None)
    sq_oof = np.clip(sq_oof, 0, None)
    id_test = np.clip(id_test, 0, None)
    sq_test = np.clip(sq_test, 0, None)

    for w in w_grid:
        final_oof = np.clip(w * id_oof + (1 - w) * sq_oof, 0, None)

        # eps 탐색
        for eps in eps_grid:
            oof_int = rule_adjust(final_oof, sq_oof, eps)
            score = rmse(y, oof_int)
            if score < best["rmse"]:
                best = {"rmse": float(score), "alpha_id": float(alpha_id), "w": float(w), "eps": float(eps)}
                best_artifacts = (id_oof.copy(), sq_oof.copy(), id_test.copy(), sq_test.copy())

print("\n BEST params:", best)

# =========================
# 7) BEST로 test 예측 + 저장
# =========================
id_oof, sq_oof, id_test, sq_test = best_artifacts
w = best["w"]
eps = best["eps"]

final_oof = np.clip(w * id_oof + (1 - w) * sq_oof, 0, None)
final_test = np.clip(w * id_test + (1 - w) * sq_test, 0, None)

oof_int = rule_adjust(final_oof, sq_oof, eps)
test_int = rule_adjust(final_test, sq_test, eps)

print("OOF RMSE (rule-adjust):", rmse(y, oof_int))
print("Test stats mean/min/max:", float(test_int.mean()), float(test_int.min()), float(test_int.max()))

# sub = pd.read_csv("sample_submission.csv")
# sub["Calories_Burned"] = test_int.astype(int)
# out_name = "submit_top5_retuned_rule_safe.csv"
# sub.to_csv(out_name, index=False)
# print(" saved:", out_name)

TOP5 exists: ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']
TOP5 missing: []
Final train/test shape: (7500, 34) (7500, 28)
Reduced train/test shape: (7500, 22) (7500, 22)


alpha_id search: 100%|██████████| 41/41 [01:46<00:00,  2.60s/it]


✅ BEST params: {'rmse': 0.21878452108562588, 'alpha_id': 0.078, 'w': 0.9875, 'eps': 0.0014}
OOF RMSE (rule-adjust): 0.21878452108562588
Test stats mean/min/max: 18.612133333333333 0.0 61.0
✅ saved: submit_top5_retuned_rule_safe.csv


정수 타겟 특성을 반영해 라운딩을 적용했고, 이후 0.5 경계 근처(±0.0014)에서만 보조 예측으로 tie-break하는 규칙 기반 보정으로 0.2188까지 개선

## rule-adjust (eps=0.004): 0.21848 여기서 더 강화해볼건 없어?

In [70]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def prep_df(df):
    d = df.copy()
    # 단위 변환
    d['Temp_diff']   = d['Body_Temperature(F)'] - 98.6
    d['Weight_kg']   = d['Weight(lb)'] * 0.453592      # lb → kg
    d['Duration_hr'] = d['Exercise_Duration'] / 60     # 분 → 시간
    # 칼로리 공식 기반 핵심 interaction
    d['DB']    = d['Exercise_Duration'] * d['BPM']
    d['DB_T']  = d['DB'] * d['Temp_diff']
    d['DB_A']  = d['DB'] * d['Age']
    d['DB_W']  = d['DB'] * d['Weight_kg']
    d['DB_TA'] = d['DB'] * d['Temp_diff'] * d['Age']
    d['DW']    = d['Exercise_Duration'] * d['Weight_kg']
    d['BT']    = d['BPM'] * d['Temp_diff']
    d['WT']    = d['Weight_kg'] * d['Temp_diff']
    d['AT']    = d['Age'] * d['Temp_diff']
    return d

# Height 제외 — OOF 검증에서 오히려 나빠짐
CORE = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight_kg',
        'Duration_hr', 'DB', 'DB_T', 'DB_A', 'DB_W', 'DB_TA',
        'DW', 'BT', 'WT', 'AT']

train_prep = prep_df(train)
test_prep  = prep_df(test)

male_tr  = train_prep['Gender'] == 'M'
female_tr = ~male_tr
male_te  = test_prep['Gender'] == 'M'
female_te = ~male_te

# ── OOF 검증 ──────────────────────────────────────
oof_all = np.zeros(len(train))
for gender, mask in [('M', male_tr), ('F', female_tr)]:
    X_g = train_prep.loc[mask, CORE].values
    y_g = train_prep.loc[mask, 'Calories_Burned'].values
    oof_g = np.zeros(len(y_g))
    for tr_idx, va_idx in kf.split(X_g):
        poly = PolynomialFeatures(degree=3, include_bias=False)
        X_tr_p = poly.fit_transform(X_g[tr_idx])
        X_va_p = poly.transform(X_g[va_idx])
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr_p)
        X_va_s = sc.transform(X_va_p)
        m = RidgeCV(alphas=np.logspace(-4, 3, 200))
        m.fit(X_tr_s, y_g[tr_idx])
        oof_g[va_idx] = m.predict(X_va_s)
    oof_all[mask.values] = oof_g

print(f"OOF float: {rmse(train['Calories_Burned'].values, oof_all):.4f}")
print(f"OOF round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all,0,None))):.4f}")

# ── 전체 학습 후 test 예측 ─────────────────────────
test_pred = np.zeros(len(test))
for gender, mask, tmask in [('M', male_tr, male_te), ('F', female_tr, female_te)]:
    X_tr = train_prep.loc[mask, CORE].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[tmask, CORE].values

    poly = PolynomialFeatures(degree=3, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)

    m = RidgeCV(alphas=np.logspace(-4, 3, 200))
    m.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  alpha={m.alpha_:.5f}  train_round={rmse(y_tr, np.rint(np.clip(m.predict(X_tr_s),0,None))):.4f}")
    test_pred[tmask.values] = m.predict(X_te_s)

# test_round = np.rint(np.clip(test_pred, 0, None)).astype(int)
# sub = pd.read_csv("sample_submission.csv")
# sub['Calories_Burned'] = test_round
# sub.to_csv("submit_interaction_deg3.csv", index=False)
# print(f"\n저장: submit_interaction_deg3.csv")
# print(f"test mean/min/max: {test_round.mean():.1f} / {test_round.min()} / {test_round.max()}")

OOF float: 0.2979
OOF round: 0.2123
Gender=M  alpha=0.00255  train_round=0.1579
Gender=F  alpha=0.00732  train_round=0.1562

저장: submit_interaction_deg3.csv
test mean/min/max: 89.7 / 1 / 314


In [71]:
# degree별 OOF 비교 (3,4,5)
for deg in [3, 4, 5]:
    oof_all = np.zeros(len(train))
    for gender, mask in [('M', male_tr), ('F', female_tr)]:
        X_g = train_prep.loc[mask, CORE].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        oof_g = np.zeros(len(y_g))
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            m = RidgeCV(alphas=np.logspace(-4, 3, 200))
            m.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = m.predict(X_va_s)
        oof_all[mask.values] = oof_g
    print(f"deg={deg}  OOF round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all,0,None))):.4f}")

deg=3  OOF round: 0.2123
deg=4  OOF round: 0.2309
deg=5  OOF round: 0.2452


In [72]:
# interaction 피처 없이 원본 변수만 / 있을 때 비교
# 그리고 Weight_kg 대신 Weight(lb) 원본이 더 나은지 확인

col_sets = {
    '원본5개':        ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)'],
    'kg변환':         ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight_kg'],
    '원본+interaction': ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)',
                        'DB', 'DB_T', 'DB_A', 'DB_W', 'DB_TA', 'DW', 'BT', 'WT', 'AT'],
    'kg+interaction': CORE,
    'Duration_hr만':  ['Duration_hr', 'BPM', 'Temp_diff', 'Age', 'Weight_kg',
                        'DB', 'DB_T', 'DB_A', 'DB_W', 'DB_TA', 'DW', 'BT', 'WT', 'AT'],
}

for name, cols in col_sets.items():
    oof_all = np.zeros(len(train))
    for gender, mask in [('M', male_tr), ('F', female_tr)]:
        X_g = train_prep.loc[mask, cols].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        oof_g = np.zeros(len(y_g))
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=3, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            m = RidgeCV(alphas=np.logspace(-4, 3, 200))
            m.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = m.predict(X_va_s)
        oof_all[mask.values] = oof_g
    r = rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all, 0, None)))
    print(f"{name}: OOF round {r:.4f}")

원본5개: OOF round 0.1740
kg변환: OOF round 0.1740
원본+interaction: OOF round 0.2157
kg+interaction: OOF round 0.2123
Duration_hr만: OOF round 0.2157


In [73]:
BEST_COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

test_pred = np.zeros(len(test))
for gender, mask, tmask in [('M', male_tr, male_te), ('F', female_tr, female_te)]:
    X_tr = train_prep.loc[mask, BEST_COLS].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[tmask, BEST_COLS].values

    poly = PolynomialFeatures(degree=3, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)

    m = RidgeCV(alphas=np.logspace(-4, 3, 200))
    m.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  alpha={m.alpha_:.5f}  "
          f"train_round={rmse(y_tr, np.rint(np.clip(m.predict(X_tr_s),0,None))):.4f}")
    test_pred[tmask.values] = m.predict(X_te_s)

test_round = np.rint(np.clip(test_pred, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_round
sub.to_csv("submit_best5_deg3.csv", index=False)
print(f"\nOOF round: 0.1740  →  저장: submit_best5_deg3.csv")

Gender=M  alpha=0.00010  train_round=0.1299
Gender=F  alpha=0.00010  train_round=0.1392

OOF round: 0.1740  →  저장: submit_best5_deg3.csv


In [54]:
# test 누수 없는지 확인
# PolynomialFeatures: train만으로 fit → test는 transform만 ✅
# StandardScaler 썼다면: train만으로 fit → test는 transform만 ✅

# 아까 deg=5 lstsq 버전은 StandardScaler 없었으니 문제없음
# 바로 제출 가능한 코드 재실행
for gender, mask, test_mask in [('M', male_mask2, male_test), ('F', female_mask2, female_test)]:
    X_tr = train_prep.loc[mask, core_cols].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[test_mask, core_cols].values

    poly = PolynomialFeatures(degree=5, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)   # train만 fit
    X_te_p = poly.transform(X_te)       # test는 transform만

    X_tr_b = np.column_stack([X_tr_p, np.ones(len(X_tr_p))])
    X_te_b = np.column_stack([X_te_p, np.ones(len(X_te_p))])

    c, _, _, _ = lstsq(X_tr_b, y_tr, rcond=None)
    test_pred_all[test_mask.values] = X_te_b @ c

test_pred_round = np.rint(np.clip(test_pred_all, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_pred_round
sub.to_csv("submit_poly5_final.csv", index=False)
print("저장 완료!")
print(f"train RMSE round: {rmse(train['Calories_Burned'].values, np.rint(np.clip(pred_all,0,None))):.4f}")

저장 완료!
train RMSE round: 0.1717


# 리더보드 점수 : 0.15318

In [74]:
# degree 4, 5 OOF 비교
BEST_COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

for deg in [3, 4, 5]:
    oof_all = np.zeros(len(train))
    for gender, mask in [('M', male_tr), ('F', female_tr)]:
        X_g = train_prep.loc[mask, BEST_COLS].values
        y_g = train_prep.loc[mask, 'Calories_Burned'].values
        oof_g = np.zeros(len(y_g))
        for tr_idx, va_idx in kf.split(X_g):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            X_tr_p = poly.fit_transform(X_g[tr_idx])
            X_va_p = poly.transform(X_g[va_idx])
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(X_tr_p)
            X_va_s = sc.transform(X_va_p)
            m = RidgeCV(alphas=np.logspace(-4, 3, 200))
            m.fit(X_tr_s, y_g[tr_idx])
            oof_g[va_idx] = m.predict(X_va_s)
        oof_all[mask.values] = oof_g
    r = rmse(train['Calories_Burned'].values, np.rint(np.clip(oof_all, 0, None)))
    print(f"deg={deg}  OOF round: {r:.4f}")

deg=3  OOF round: 0.1740
deg=4  OOF round: 0.2017
deg=5  OOF round: 0.2126


In [75]:
# alpha 범위 더 세밀하게 + 성별별 최적 alpha 확인
BEST_COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

for gender, mask in [('M', male_tr), ('F', female_tr)]:
    X_g = train_prep.loc[mask, BEST_COLS].values
    y_g = train_prep.loc[mask, 'Calories_Burned'].values
    oof_g = np.zeros(len(y_g))
    best_alphas = []
    for tr_idx, va_idx in kf.split(X_g):
        poly = PolynomialFeatures(degree=3, include_bias=False)
        X_tr_p = poly.fit_transform(X_g[tr_idx])
        X_va_p = poly.transform(X_g[va_idx])
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr_p)
        X_va_s = sc.transform(X_va_p)
        m = RidgeCV(alphas=np.logspace(-6, 4, 500))  # 더 넓고 촘촘하게
        m.fit(X_tr_s, y_g[tr_idx])
        oof_g[va_idx] = m.predict(X_va_s)
        best_alphas.append(m.alpha_)
    r = rmse(y_g, np.rint(np.clip(oof_g, 0, None)))
    print(f"Gender={gender}  OOF round: {r:.4f}  alphas: {[f'{a:.5f}' for a in best_alphas]}")

Gender=M  OOF round: 0.1693  alphas: ['0.00002', '0.00001', '0.00002', '0.00001', '0.00001']
Gender=F  OOF round: 0.1724  alphas: ['0.00003', '0.00002', '0.00008', '0.00003', '0.00002']


In [76]:
BEST_COLS = ['Exercise_Duration', 'BPM', 'Temp_diff', 'Age', 'Weight(lb)']

test_pred = np.zeros(len(test))
for gender, mask, tmask in [('M', male_tr, male_te), ('F', female_tr, female_te)]:
    X_tr = train_prep.loc[mask, BEST_COLS].values
    y_tr = train_prep.loc[mask, 'Calories_Burned'].values
    X_te = test_prep.loc[tmask, BEST_COLS].values

    poly = PolynomialFeatures(degree=3, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr)
    X_te_p = poly.transform(X_te)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr_p)
    X_te_s = sc.transform(X_te_p)

    m = RidgeCV(alphas=np.logspace(-6, 4, 500))
    m.fit(X_tr_s, y_tr)
    print(f"Gender={gender}  alpha={m.alpha_:.6f}  "
          f"train_round={rmse(y_tr, np.rint(np.clip(m.predict(X_tr_s),0,None))):.4f}")
    test_pred[tmask.values] = m.predict(X_te_s)

test_round = np.rint(np.clip(test_pred, 0, None)).astype(int)
sub = pd.read_csv("sample_submission.csv")
sub['Calories_Burned'] = test_round
sub.to_csv("submit_best5_deg3_fineralpha.csv", index=False)
print("\n저장: submit_best5_deg3_fineralpha.csv")

Gender=M  alpha=0.000015  train_round=0.1289
Gender=F  alpha=0.000026  train_round=0.1392

저장: submit_best5_deg3_fineralpha.csv


## submit_best5_deg3_fineralpha 아직 제출 안해봄...